# Paper 20: Concentration and sector lid

v2 is the same FCF list as papers 16–19, cap-weighted, with a **10% name cap** and **no sector cap**. On the last honest session IT is about **66%** of the invested book and the top five names are about **48%**. Paper 08 already chose the 10% name lid. This paper asks whether a **sector ceiling** (and a tighter 8% name lid) steadies Calmar / max drawdown, or whether it just chops the FCF winners.

This is a **ceiling**, not a floor. Excess IT weight is scaled down and spread over the other sectors already on the FCF list. It does **not** pull in banned businesses, and it does **not** set minimums on health care, industrials, or staples — that is paper 25.

Change only `construction`. Brain, overlay, execution, and screens stay on v2. Scored against **v2 baseline**. Kill rules are frozen in `Research/book-construction-status.md`:

- Train 2022 calendar more than 5pp worse than SPUS
- Train max drawdown worse than SPUS
- Train Calmar below both v2 baseline and SPUS
- Test 2023 calendar more than 15pp worse than v2 baseline

**Variants.** sector-40, sector-50, name-8, sector-40-name-8.


In [1]:
import subprocess
import sys
from pathlib import Path

def _repo_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "monterey" / "__init__.py").exists():
            return candidate
    raise FileNotFoundError(f"Monterey repo root not found from {start}")

_ROOT = _repo_root()
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))
try:
    import monterey  # noqa: F401
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", f"{_ROOT}[research]"])

from monterey.research import boot, compare, run_book
from monterey.sprints import score_against_baseline
from monterey.spec import BookSpec

rd = boot("paper-20-concentration", start="2019-12-01", end="2026-09-30")
print("snapshot", rd.manifest.get("hash"))
print("v2 hash", BookSpec.load("fcf-sma-v2-baseline").hash())

base = run_book(
    "fcf-sma-v2-baseline",
    rd,
    paper="20-concentration",
    id="v2",
)


[monterey.data] Loading snapshot ebbbefb35545


        field                   value
       window 2019-12-01 → 2026-09-30
     universe    sp500, point-in-time
      symbols                     631
    snapshots                      85
   price rows                 1147266
    dividends                    9754
filing events                   61274
   halalquant                   0.4.0
snapshot hash            ebbbefb35545
snapshot ebbbefb35545
v2 hash 6ca86844209e


## One lid at a time

Each trial copies v2 and changes only `construction`. The FCF list is the same. The overlay is still SPY 200-day. A 40% sector lid is the harder ceiling; 50% is closer to what a Halal mega-cap book already looks like. `name_cap=0.08` is a tighter single-name lid on top of paper 08's 10%. The last trial stacks 40% + 8% so we can see whether the two lids fight each other when leftover weight has nowhere to go.


In [2]:
sector_40 = run_book(
    "fcf-sma-v2-baseline",
    rd,
    paper="20-concentration",
    id="sector-40",
    construction={"sector_cap": 0.40},
)
sector_50 = run_book(
    "fcf-sma-v2-baseline",
    rd,
    paper="20-concentration",
    id="sector-50",
    construction={"sector_cap": 0.50},
)
name_8 = run_book(
    "fcf-sma-v2-baseline",
    rd,
    paper="20-concentration",
    id="name-8",
    construction={"name_cap": 0.08},
)
both = run_book(
    "fcf-sma-v2-baseline",
    rd,
    paper="20-concentration",
    id="sector-40-name-8",
    construction={"sector_cap": 0.40, "name_cap": 0.08},
)

trials = [sector_40, sector_50, name_8, both]
for book in [base, *trials]:
    print(book.spec.id, book.spec.hash())
    print(" ", dict(book.spec.summary()))


v2 6ca86844209e
  {'book': 'v2', 'hash': '6ca86844209e', 'universe': 'sp500 point-in-time', 'sector screen': 'on', 'AAOIFI lines': 'debt 30% / cash 30% / recv 70%', 'breach exit': 'next_open', 'brain': 'fcf_quality (keep_quantile=0.5, min_holdings=20)', 'weights': 'cap', 'name cap': '10%', 'sector cap': 'none', 'throttle': 'SPY 200-day SMA', 'rebalance': 'daily_to_target, drift band 0.0%, min $100', 'dividends': 'credited', 'cost': '10 bp charged', 'purify': 'ex_date'}
sector-40 147feb07242d
  {'book': 'sector-40', 'hash': '147feb07242d', 'universe': 'sp500 point-in-time', 'sector screen': 'on', 'AAOIFI lines': 'debt 30% / cash 30% / recv 70%', 'breach exit': 'next_open', 'brain': 'fcf_quality (keep_quantile=0.5, min_holdings=20)', 'weights': 'cap', 'name cap': '10%', 'sector cap': '40%', 'throttle': 'SPY 200-day SMA', 'rebalance': 'daily_to_target, drift band 0.0%, min $100', 'dividends': 'credited', 'cost': '10 bp charged', 'purify': 'ex_date'}
sector-50 37ba1d34a860
  {'book': 'sect

In [3]:
compare([base, *trials], rd)


,total_return,cagr,volatility,max_drawdown,calmar,sharpe,sortino,turnover,costs_pct,dividends_pct,avg_exposure,switches,hash
book,,,,,,,,,,,,,
v2,1.0644,0.1137,0.1463,-0.2482,0.4582,0.6752,0.8589,4.1653,0.0561,0.0561,0.8032,36.0000,6ca86844209e
sector-40,0.9561,0.1048,0.1403,-0.2475,0.4236,0.6406,0.8103,4.2227,0.0568,0.0586,0.8032,36.0000,147feb07242d
sector-50,1.0176,0.1099,0.1442,-0.2482,0.4429,0.6590,0.8350,4.1820,0.0563,0.0569,0.8032,36.0000,37ba1d34a860
name-8,0.9842,0.1072,0.1427,-0.2523,0.4249,0.6472,0.8217,4.2471,0.0572,0.0593,0.8032,36.0000,dd28de1d71d8
sector-40-name-8,0.8815,0.0985,0.1371,-0.2523,0.3904,0.6101,0.7690,4.2908,0.0578,0.0618,0.8032,36.0000,d0a51ac542c5
SPUS (total return),2.1167,0.1840,0.2116,-0.3080,0.5975,0.8123,1.0828,NaN,NaN,NaN,NaN,NaN,NaN
SPY (total return),1.6113,0.1533,0.2006,-0.3372,0.4547,0.7140,0.8854,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
for trial in trials:
    verdict = score_against_baseline(trial, base, rd)
    reasons = "; ".join(verdict["reasons"]) or "pass"
    print(trial.spec.id, "KILLED" if verdict["killed"] else "keep", reasons)


sector-40 keep pass
sector-50 keep pass
name-8 KILLED train Calmar below both v2 baseline and SPUS
sector-40-name-8 KILLED train Calmar below both v2 baseline and SPUS


## Did the lid bind?

`compare` only sees NAV. Before we trust a Calmar move, rebuild the fully invested weights on a few month-ends and check that IT actually hits the ceiling and that no name sits above the name cap. If every sector would breach the lid, `apply_sector_cap` keeps the last pass and the lid is not met.


In [5]:
import pandas as pd
from monterey.layers import screen, build_weights
from monterey.layers.signals import Context, run_brain
from monterey.spec import BookSpec

DAYS = ["2020-03-31", "2022-06-30", "2023-12-29", "2024-12-31"]

SPECS = {
    "v2": BookSpec.load("fcf-sma-v2-baseline"),
    "sector-40": BookSpec.load("fcf-sma-v2-baseline").replace(
        id="sector-40", construction={"sector_cap": 0.40}
    ),
    "sector-50": BookSpec.load("fcf-sma-v2-baseline").replace(
        id="sector-50", construction={"sector_cap": 0.50}
    ),
    "name-8": BookSpec.load("fcf-sma-v2-baseline").replace(
        id="name-8", construction={"name_cap": 0.08}
    ),
    "sector-40-name-8": BookSpec.load("fcf-sma-v2-baseline").replace(
        id="sector-40-name-8", construction={"sector_cap": 0.40, "name_cap": 0.08}
    ),
}


def weights_on(spec, day: str) -> pd.Series:
    held = screen(rd, day, spec)
    picked = run_brain(Context(data=rd, spec=spec, as_of=pd.Timestamp(day).date(), screen=held))
    return build_weights(picked, spec, rd.sectors)


def sector_shares(w: pd.Series) -> pd.Series:
    labels = w.index.map(lambda s: rd.sectors.get(str(s), "unknown"))
    return w.groupby(labels).sum().sort_values(ascending=False)


def it_weight(sectors: pd.Series) -> float:
    if sectors.empty:
        return 0.0
    mask = sectors.index.astype(str).str.lower().str.contains("information technology")
    return float(sectors[mask].sum())


rows = []
for day in DAYS:
    for name, spec in SPECS.items():
        w = weights_on(spec, day)
        sectors = sector_shares(w) if not w.empty else pd.Series(dtype=float)
        it = it_weight(sectors)
        cap_name = spec.construction.get("name_cap")
        cap_sec = spec.construction.get("sector_cap")
        rows.append(
            {
                "day": day,
                "book": name,
                "n": int(len(w)),
                "max_name": float(w.max()) if len(w) else None,
                "names_at_cap": int((w >= (cap_name or 1.0) - 1e-4).sum()) if cap_name else 0,
                "it": it,
                "top_sector": sectors.index[0] if len(sectors) else None,
                "top_sector_w": float(sectors.iloc[0]) if len(sectors) else None,
                "sector_over_lid": int((sectors > (cap_sec or 1.0) + 1e-9).sum()) if cap_sec else 0,
            }
        )

census = pd.DataFrame(rows)
census


,day,book,n,max_name,names_at_cap,it,top_sector,top_sector_w,sector_over_lid
0,2020-03-31,v2,82,0.1000,1,0.2982,information technology,0.2982,0
1,2020-03-31,sector-40,82,0.1000,1,0.2982,information technology,0.2982,0
2,2020-03-31,sector-50,82,0.1000,1,0.2982,information technology,0.2982,0
3,2020-03-31,name-8,82,0.0800,2,0.2850,information technology,0.2850,0
4,2020-03-31,sector-40-name-8,82,0.0800,2,0.2850,information technology,0.2850,0
5,2022-06-30,v2,98,0.1000,2,0.3612,information technology,0.3612,0
6,2022-06-30,sector-40,98,0.1000,2,0.3612,information technology,0.3612,0
7,2022-06-30,sector-50,98,0.1000,2,0.3612,information technology,0.3612,0
8,2022-06-30,name-8,98,0.0800,2,0.3293,information technology,0.3293,0
9,2022-06-30,sector-40-name-8,98,0.0800,2,0.3293,information technology,0.3293,0


In [6]:
day = "2024-12-31"
v2_w = weights_on(SPECS["v2"], day)
lid_w = weights_on(SPECS["sector-40"], day)

print("hash v2", SPECS["v2"].hash(), "sector-40", SPECS["sector-40"].hash())
print("v2 names", len(v2_w), "sector-40 names", len(lid_w))
print("same FCF list", set(v2_w.index) == set(lid_w.index))

mix = pd.concat(
    {
        "v2": sector_shares(v2_w),
        "sector-40": sector_shares(lid_w),
        "sector-50": sector_shares(weights_on(SPECS["sector-50"], day)),
        "name-8": sector_shares(weights_on(SPECS["name-8"], day)),
        "sector-40-name-8": sector_shares(weights_on(SPECS["sector-40-name-8"], day)),
    },
    axis=1,
).fillna(0.0)
display(mix)

print("largest names on", day)
names = pd.concat(
    {
        "v2": v2_w.sort_values(ascending=False).head(8),
        "sector-40": lid_w.sort_values(ascending=False).head(8),
        "name-8": weights_on(SPECS["name-8"], day).sort_values(ascending=False).head(8),
    },
    axis=1,
)
display(names)


hash v2 6ca86844209e sector-40 147feb07242d
v2 names 98 sector-40 names 98
same FCF list True


,v2,sector-40,sector-50,name-8,sector-40-name-8
symbol,,,,,
information technology,0.4939,0.4000,0.4939,0.4782,0.4000
communication services,0.1906,0.2181,0.1906,0.1770,0.1817
consumer discretionary,0.1406,0.1531,0.1406,0.1299,0.1437
financials,0.0575,0.0753,0.0575,0.0707,0.0903
health care,0.0548,0.0717,0.0548,0.0673,0.0860
industrials,0.0330,0.0431,0.0330,0.0405,0.0517
consumer staples,0.0248,0.0325,0.0248,0.0305,0.0390
materials,0.0042,0.0055,0.0042,0.0051,0.0066
technology,0.0006,0.0008,0.0006,0.0007,0.0009


largest names on 2024-12-31


,v2,sector-40,name-8
symbol,,,
AAPL,0.1000,0.0810,0.0800
NVDA,0.1000,0.0810,0.0800
MSFT,0.1000,0.0810,0.0800
GOOG,0.1000,0.1000,0.0800
AMZN,0.1000,0.1000,0.0800
META,0.0768,0.1000,0.0800
AVGO,0.0548,0.0444,0.0673
MA,0.0246,0.0321,0.0302


## Path, not just the scorecard

Kill rules decide keep vs killed. They do not show how the books made or lost money. Plot **NAV** (growth of $1) and **drawdown**. Then plot **IT invested weight** — that is the quantity this lid claims to change. A name-only trial should barely move IT; a 40% lid should pin it.


In [7]:
import matplotlib.pyplot as plt
from monterey.diagnostics import yearly

books = [base, *trials]
figures = Path("figures")
figures.mkdir(exist_ok=True)

spus = rd.total_return("SPUS").reindex(base.nav_series().index).ffill()

fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
for book in books:
    nav = book.nav_series()
    growth = nav / nav.iloc[0]
    axes[0].plot(growth.index, growth, label=book.spec.id)
    dd = nav / nav.cummax() - 1.0
    axes[1].plot(dd.index, dd, label=book.spec.id)

axes[0].plot(spus.index, spus / spus.iloc[0], label="SPUS", alpha=0.7)
axes[0].set_title("Growth of $1")
axes[0].set_ylabel("NAV / start")
axes[0].legend()
axes[1].set_title("Drawdown from peak")
axes[1].set_ylabel("Drawdown")
axes[1].legend()

fig.tight_layout()
fig.savefig(figures / "nav-and-drawdown.png", dpi=140)
plt.show()
print("wrote", figures / "nav-and-drawdown.png")


wrote figures/nav-and-drawdown.png


/var/folders/lf/wzrn2w4j7mq_hf01km30xn0c0000gn/T/ipykernel_80778/1682924597.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
def invested_it_weight(ledger) -> pd.Series:
    pos = ledger.positions.copy()
    pos["date"] = pd.to_datetime(pos["date"])
    pos["invested_weight"] = pos["invested_weight"].astype(float)
    live = pos[pos["invested_weight"] > 0]
    if live.empty:
        return pd.Series(dtype=float)
    tot = live.groupby("date")["invested_weight"].sum()
    it = live[live["sector"].fillna("").str.lower().str.contains("information technology")]
    return it.groupby("date")["invested_weight"].sum().reindex(tot.index).fillna(0.0) / tot

fig, ax = plt.subplots(figsize=(10, 4))
for book in books:
    series = invested_it_weight(book)
    ax.plot(series.index, series, label=book.spec.id)
ax.axhline(0.40, color="black", ls="--", lw=1, label="40% lid")
ax.axhline(0.50, color="gray", ls=":", lw=1, label="50% lid")
ax.set_title("Information Technology invested weight")
ax.set_ylabel("Share of the stock book")
ax.legend()
fig.tight_layout()
fig.savefig(figures / "it-weight.png", dpi=140)
plt.show()
print("wrote", figures / "it-weight.png")


wrote figures/it-weight.png


/var/folders/lf/wzrn2w4j7mq_hf01km30xn0c0000gn/T/ipykernel_80778/959441789.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
calendar = pd.concat(
    {book.spec.id: yearly(book.nav_series())["return"] for book in books},
    axis=1,
)
calendar["SPUS"] = yearly(spus)["return"]
display(calendar)

ax = calendar.plot(kind="bar", figsize=(10, 4))
ax.set_title("Calendar-year return")
ax.set_ylabel("Return")
ax.axhline(0, color="black", lw=0.8)
fig = ax.get_figure()
fig.tight_layout()
fig.savefig(figures / "calendar-returns.png", dpi=140)
plt.show()
print("wrote", figures / "calendar-returns.png")


,v2,sector-40,sector-50,name-8,sector-40-name-8,SPUS
year,,,,,,
2020,0.0528,0.0528,0.0528,0.0496,0.0496,0.2399
2021,0.2831,0.2830,0.2831,0.2759,0.2759,0.3562
2022,-0.2080,-0.2076,-0.2080,-0.2090,-0.2090,-0.2276
2023,0.1525,0.1525,0.1525,0.1395,0.1394,0.3420
2024,0.2741,0.2636,0.2741,0.2528,0.2438,0.2649
2025,0.1250,0.1209,0.1209,0.1210,0.1176,0.1978
2026,0.1681,0.1196,0.1458,0.1706,0.1214,0.1802


wrote figures/calendar-returns.png


/var/folders/lf/wzrn2w4j7mq_hf01km30xn0c0000gn/T/ipykernel_80778/2111898560.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Construction side effects

A sector lid can still be the same mega-cap list with dollars moved out of IT. Read positions for last-session mix, top-five weight, effective names, and turnover. `compare` cannot see those.


In [10]:
from monterey.diagnostics import diagnose
from monterey.diagnostics.exposure import concentration, sector_mix


def live_book(ledger) -> pd.DataFrame:
    pos = ledger.positions.copy()
    pos["date"] = pd.to_datetime(pos["date"])
    pos["invested_weight"] = pos["invested_weight"].astype(float)
    return pos[pos["invested_weight"] > 0]


def it_share(live: pd.DataFrame) -> float:
    if live.empty:
        return float("nan")
    tot = live.groupby("date")["invested_weight"].sum()
    it = live[live["sector"].fillna("").str.lower().str.contains("information technology")]
    daily = it.groupby("date")["invested_weight"].sum().reindex(tot.index).fillna(0.0) / tot
    return float(daily.mean())


def top5_share(live: pd.DataFrame) -> float:
    if live.empty:
        return float("nan")
    daily = (
        live.sort_values(["date", "invested_weight"], ascending=[True, False])
        .groupby("date")["invested_weight"]
        .apply(lambda s: float(s.head(5).sum()))
    )
    tot = live.groupby("date")["invested_weight"].sum()
    return float((daily / tot).mean())


side = []
last_mix = {}
for book in books:
    live = live_book(book)
    last = live[live["date"] == live["date"].max()] if not live.empty else live
    diag = diagnose(book, rd, write=True)
    conc = concentration(book.positions, book.spec.construction.get("name_cap"))
    mix = sector_mix(book.positions)
    last_mix[book.spec.id] = mix
    side.append(
        {
            "book": book.spec.id,
            "mean_targets": float(book.nav["n_targets"].mean()),
            "mean_it": it_share(live),
            "mean_top5": top5_share(live),
            "last_it": float(
                mix[mix.index.astype(str).str.lower().str.contains("information technology")].sum()
            ) if not mix.empty else None,
            "last_n": int(last["symbol"].nunique()) if not last.empty else 0,
            "effective_names": float(conc["effective_names"].mean()) if not conc.empty else None,
            "turnover": diag.get("trading", {}).get("annual_turnover") or diag.get("trading", {}).get("turnover"),
            "hash": book.spec.hash(),
        }
    )

display(pd.DataFrame(side).set_index("book"))
display(pd.concat(last_mix, axis=1).fillna(0.0))


,mean_targets,mean_it,mean_top5,last_it,last_n,effective_names,turnover,hash
book,,,,,,,,
v2,76.2979,0.4376,0.4193,0.6557,115,21.9525,4.1653,6ca86844209e
sector-40,76.2979,0.3846,0.3961,0.3956,115,23.5675,4.2227,147feb07242d
sector-50,76.2979,0.4199,0.4125,0.4951,115,22.4607,4.1820,37ba1d34a860
name-8,76.2979,0.4163,0.3564,0.6234,115,27.0584,4.2471,dd28de1d71d8
sector-40-name-8,76.2979,0.3740,0.3383,0.3949,115,28.7872,4.2908,d0a51ac542c5


,v2,sector-40,sector-50,name-8,sector-40-name-8
sector,,,,,
information technology,0.6557,0.3956,0.4951,0.6234,0.3949
communication services,0.0942,0.1525,0.1382,0.1031,0.1346
health care,0.0763,0.1379,0.1119,0.0834,0.1435
industrials,0.0613,0.1108,0.0899,0.0670,0.1153
financials,0.0496,0.0896,0.0727,0.0543,0.0933
consumer discretionary,0.0329,0.0595,0.0483,0.0360,0.0619
consumer staples,0.0193,0.0348,0.0282,0.0211,0.0362
materials,0.0082,0.0147,0.0120,0.0089,0.0154
energy,0.0026,0.0046,0.0037,0.0028,0.0048
